In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.analise_acoes AS

-- 1. COTAÇÃO MAIS RECENTE
-- Para cada ação, pega somente a cotação da data mais recente.
WITH cotacao_recente AS (
    SELECT
        symbol,
        date,
        close
    FROM workspace.silver.cotacoes

    -- ROW_NUMBER cria uma numeração por ação.
    -- A data mais recente recebe número 1.
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY symbol
        ORDER BY date DESC
    ) = 1
),

-- 2. ESTATÍSTICAS MAIS RECENTES DA BRAPI

-- A BRAPI já fornece P/L, P/VP, VPA e LPA.
-- Portanto, usamos esses indicadores diretamente.
estatisticas_recentes AS (
    SELECT
        symbol,
        date,
        bookValue,
        priceToBook,
        trailingEps,
        trailingPE
    FROM workspace.silver.estatisticas

    -- Mantém somente o registro mais recente de cada ação.
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY symbol
        ORDER BY date DESC
    ) = 1
),

-- 3. DADOS FINANCEIROS MAIS RECENTES

-- Aqui pegamos os indicadores financeiros já fornecidos pela BRAPI.
financeiro_recente AS (
    SELECT
        symbol,
        date,
        totalCash,
        ebitda,
        totalDebt,
        returnOnEquity,
        profitMargins
    FROM workspace.silver.dados_financeiros

    -- Mantém o registro financeiro mais recente de cada ação.
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY symbol
        ORDER BY date DESC
    ) = 1
)

-- 4. MONTAGEM DA TABELA GOLD

SELECT

    -- Identificação da ação
    c.symbol AS codigo_acao,

    -- Preço e data da cotação mais recente
    c.close AS preco_atual,
    c.date AS data_preco,

    -- Indicadores que já vieram calculados pela BRAPI
    e.trailingPE AS pl,
    e.bookValue AS valor_patrimonial,
    e.priceToBook AS pvp,
    e.trailingEps AS eps,

    -- Indicadores financeiros da BRAPI
    f.profitMargins AS margem_liquida,
    f.returnOnEquity AS roe,
    f.ebitda,

    -- Valores necessários para calcular dívida líquida
    f.totalDebt AS divida_total,
    f.totalCash AS caixa,

    -- Dívida líquida = dívida total - caixa
    f.totalDebt - f.totalCash AS divida_liquida


-- A cotação é nossa tabela principal.
FROM cotacao_recente c

-- Junta os indicadores de mercado pelo código da ação.
LEFT JOIN estatisticas_recentes e
    ON c.symbol = e.symbol

-- Junta os dados financeiros pelo código da ação.
LEFT JOIN financeiro_recente f
    ON c.symbol = f.symbol;

In [0]:
%sql
-- Visualiza a Gold criada.
SELECT *
FROM workspace.gold.analise_acoes
ORDER BY codigo_acao;

In [0]:
%sql

-- ETAPA 2 — GOLD: DIVIDENDOS E DY DOS ÚLTIMOS 5 ANOS

CREATE OR REPLACE TABLE workspace.gold.analise_acoes AS

WITH

-- 1. PREÇO MAIS RECENTE

-- Pega o último preço disponível para cada ação.
cotacao_recente AS (
    SELECT
        symbol,
        date,
        close
    FROM workspace.silver.cotacoes

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY symbol
        ORDER BY date DESC
    ) = 1
),

-- 2. ESTATÍSTICAS MAIS RECENTES

-- Utiliza os indicadores que a BRAPI já calculou.
estatisticas_recentes AS (
    SELECT
        symbol,
        date,
        bookValue,
        priceToBook,
        trailingEps,
        trailingPE
    FROM workspace.silver.estatisticas

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY symbol
        ORDER BY date DESC
    ) = 1
),

-- 3. DADOS FINANCEIROS MAIS RECENTES

financeiro_recente AS (
    SELECT
        symbol,
        date,
        totalCash,
        ebitda,
        totalDebt,
        returnOnEquity,
        profitMargins
    FROM workspace.silver.dados_financeiros

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY symbol
        ORDER BY date DESC
    ) = 1
),


-- 4. LIMPEZA DOS EVENTOS DE PROVENTOS

-- Existem eventos com o mesmo provento e ex-date, mas diferentes datas de pagamento.
-- Para não contar o mesmo provento duas vezes, eliminamos essas duplicidades.

proventos_unicos AS (
    SELECT DISTINCT
        symbol,
        label,
        rate,
        exDate
    FROM workspace.silver.dividendos
    WHERE
        exDate IS NOT NULL
        AND rate IS NOT NULL
        AND label IN (
            'DIVIDENDO',
            'JCP',
            'RENDIMENTO'
        )
),

-- 5. PROVENTOS ANUAIS — 2021 A 2025

-- Soma todos os proventos de cada ação em cada ano.
proventos_anuais AS (
    SELECT
        symbol,
        YEAR(exDate) AS ano,
        SUM(rate) AS dividendos_ano_reais
    FROM proventos_unicos
    WHERE YEAR(exDate) BETWEEN 2021 AND 2025
    GROUP BY
        symbol,
        YEAR(exDate)
),


-- 6. PREÇO DE FIM DE CADA ANO

-- Para calcular o DY anual, usamos o último preço disponível em cada ano.
precos_fim_ano AS (
    SELECT
        symbol,
        YEAR(date) AS ano,
        close AS preco_fim_ano
    FROM workspace.silver.cotacoes
    WHERE YEAR(date) BETWEEN 2021 AND 2025

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY symbol, YEAR(date)
        ORDER BY date DESC
    ) = 1
),


-- 7. DY ANUAL

-- DY anual = dividendos pagos no ano / preço de fim do ano.
dy_anual AS (
    SELECT
        p.symbol,
        p.ano,
        p.dividendos_ano_reais,
        c.preco_fim_ano,

        CASE
            WHEN c.preco_fim_ano > 0
            THEN p.dividendos_ano_reais / c.preco_fim_ano
            ELSE NULL
        END AS dy_ano

    FROM proventos_anuais p

    LEFT JOIN precos_fim_ano c
        ON p.symbol = c.symbol
        AND p.ano = c.ano
),


-- 8. MÉDIAS DOS 5 ANOS

-- Calcula a média anual dos proventos e a média dos DY anuais.
medias_5_anos AS (
    SELECT
        symbol,

        AVG(dividendos_ano_reais)
            AS dividendos_medio_5_anos_reais,

        AVG(dy_ano) * 100
            AS dy_medio_5_anos_pct

    FROM dy_anual
    GROUP BY symbol
)


-- 9. MONTAGEM FINAL DA GOLD

SELECT

    -- Identificação
    c.symbol AS codigo_acao,

    -- Preço atual
    c.close AS preco_atual,
    c.date AS data_preco,

    -- Indicadores fornecidos pela BRAPI
    e.trailingPE AS pl,
    e.bookValue AS valor_patrimonial,
    e.priceToBook AS pvp,
    e.trailingEps AS eps,

    -- Indicadores financeiros
    f.profitMargins AS margem_liquida,
    f.returnOnEquity AS roe,
    f.ebitda,
    f.totalDebt AS divida_total,
    f.totalCash AS caixa,

    -- Indicador calculado
    f.totalDebt - f.totalCash AS divida_liquida,

    -- Indicadores históricos de proventos
    m.dividendos_medio_5_anos_reais,
    m.dy_medio_5_anos_pct

FROM cotacao_recente c

LEFT JOIN estatisticas_recentes e
    ON c.symbol = e.symbol

LEFT JOIN financeiro_recente f
    ON c.symbol = f.symbol

LEFT JOIN medias_5_anos m
    ON c.symbol = m.symbol;

In [0]:
%sql
-- Mostra os indicadores da Gold após a inclusão dos dividendos.
SELECT
    codigo_acao,
    preco_atual,
    data_preco,
    dividendos_medio_5_anos_reais,
    dy_medio_5_anos_pct
FROM workspace.gold.analise_acoes
ORDER BY codigo_acao;

In [0]:
%sql
-- Verifica como os resultados estão organizados antes de calcular o crescimento de 5 anos.

SELECT
    symbol,
    period,
    type,
    COUNT(*) AS quantidade,
    MIN(date) AS primeira_data,
    MAX(date) AS ultima_data
FROM workspace.silver.demonstracao_resultados
GROUP BY
    symbol,
    period,
    type
ORDER BY
    symbol,
    period,
    type;

In [0]:
%sql

-- STAGE 3 — CRESCIMENTO DOS LUCROS

-- SELECT:
-- Define quais informações queremos obter como resultado da consulta.

-- YEAR(date):
-- Extrai somente o ano de uma coluna que contém uma data.

-- AS:
-- Cria um nome (apelido) para o resultado da expressão.
-- Aqui, o ano calculado passa a se chamar "ano".

-- SUM(netIncome):
-- Soma os valores de lucro líquido.

-- GROUP BY:
-- Agrupa as linhas antes de realizar o SUM().
-- Neste caso, queremos uma soma separada para cada ação e para cada ano.

-- Resultado esperado:
-- uma linha para cada ação em cada ano, contendo o lucro líquido daquele ano.

WITH lucro_anual AS (

    SELECT
        symbol,
        YEAR(date) AS ano,
        SUM(netIncome) AS lucro_liquido_anual

    FROM workspace.silver.demonstracao_resultados

    -- WHERE:
    -- Filtra as linhas que serão utilizadas.

    -- Aqui mantemos somente os anos de 2021 a 2025.
    -- Também eliminamos registros sem lucro líquido.

    WHERE YEAR(date) BETWEEN 2021 AND 2025
      AND netIncome IS NOT NULL

    -- GROUP BY:
    -- Faz o SQL juntar as linhas que possuem o mesmo "symbol" e o mesmo "ano".
    
    -- Com quatro trimestres por ano, os quatro valores trimestrais serão unidos para cada ação e depois somados pelo SUM().

    GROUP BY
        symbol,
        YEAR(date)
),


lucros_5_anos AS (

    SELECT
        symbol,

        -- CASE:
        -- Permite criar uma condição dentro da consulta.
        
        -- WHEN ano = 2021:
        -- Procura somente o valor correspondente a 2021.
        
        -- THEN lucro_liquido_anual:
        -- Quando a condição for verdadeira, retorna o lucro daquele ano.
        
        -- ELSE não foi necessário aqui:
        -- quando a condição não é verdadeira, o CASE retorna NULL automaticamente.

        MAX(
            CASE
                WHEN ano = 2021
                THEN lucro_liquido_anual
            END
        ) AS lucro_2021,

        MAX(
            CASE
                WHEN ano = 2022
                THEN lucro_liquido_anual
            END
        ) AS lucro_2022,

        MAX(
            CASE
                WHEN ano = 2023
                THEN lucro_liquido_anual
            END
        ) AS lucro_2023,

        MAX(
            CASE
                WHEN ano = 2024
                THEN lucro_liquido_anual
            END
        ) AS lucro_2024,

        MAX(
            CASE
                WHEN ano = 2025
                THEN lucro_liquido_anual
            END
        ) AS lucro_2025

    FROM lucro_anual

    -- GROUP BY symbol:
    -- Queremos terminar com apenas uma linha por ação.

    -- O CASE separa cada ano em uma coluna.
    -- O MAX() transforma o valor encontrado em cada ano em um único valor que pode ser apresentado nessa mesma linha da ação.

    GROUP BY symbol
)


SELECT
    symbol AS codigo_acao,

    lucro_2021,
    lucro_2022,
    lucro_2023,
    lucro_2024,
    lucro_2025,

    -- POWER():
    -- Calcula uma potência.

    -- A fórmula abaixo representa o CAGR:
    
    -- (lucro_final / lucro_inicial)^(1/4) - 1
    
    -- O intervalo entre 2021 e 2025 possui quatro períodos
    -- de crescimento:
    -- 2021→2022, 2022→2023, 2023→2024 e 2024→2025.
    
    -- NULLIF():
    -- Evita uma divisão por zero caso lucro_2021 seja zero.
    
    -- * 100:
    -- Converte o resultado decimal para porcentagem.
    
    -- AS:
    -- Dá um nome à coluna calculada.

    CASE
        WHEN lucro_2021 > 0
         AND lucro_2025 > 0

        THEN
            (
                POWER(
                    lucro_2025 / NULLIF(lucro_2021, 0),
                    1.0 / 4
                ) - 1
            ) * 100

        ELSE NULL
    END AS crescimento_lucro_cagr_5_anos_pct

FROM lucros_5_anos;

In [0]:
%sql
-- STAGE 3 — INCORPORAR O CAGR À ANÁLISE PRINCIPAL

CREATE OR REPLACE TABLE workspace.gold.analise_acoes AS

SELECT

    -- Mantemos todos os indicadores que já estavam na tabela.

    a.codigo_acao,
    a.preco_atual,
    a.data_preco,
    a.pl,
    a.valor_patrimonial,
    a.pvp,
    a.eps,
    a.margem_liquida,
    a.roe,
    a.ebitda,
    a.divida_total,
    a.caixa,
    a.divida_liquida,

    a.dividendos_medio_5_anos_reais,
    a.dy_medio_5_anos_pct,

    -- Acrescentamos o CAGR calculado no Stage 3.

    c.crescimento_lucro_cagr_5_anos_pct

FROM workspace.gold.analise_acoes AS a

-- LEFT JOIN:
-- Combina duas tabelas.

-- Escolhemos LEFT JOIN porque queremos preservar todas as ações que já existem na nossa análise principal.

-- Assim, o ITUB4 continua presente mesmo sem CAGR. Nesse caso, o indicador ficará NULL.

LEFT JOIN workspace.gold.crescimento_lucros AS c

    -- ON:
    -- Define como as duas tabelas serão relacionadas.
    
    -- codigo_acao identifica a ação na tabela principal e também identifica a mesma ação na tabela de crescimento.

    ON a.codigo_acao = c.codigo_acao;

In [0]:
%sql

-- VERIFICAÇÃO

-- SELECT:
-- escolhe os principais campos para verificar se o CAGR entrou corretamente na tabela principal.

SELECT
    codigo_acao,
    preco_atual,
    pl,
    pvp,
    dividendos_medio_5_anos_reais,
    dy_medio_5_anos_pct,
    crescimento_lucro_cagr_5_anos_pct

FROM workspace.gold.analise_acoes

-- ORDER BY:
-- organiza as ações pelo código para facilitar a leitura.

ORDER BY codigo_acao;

In [0]:
%sql

-- STAGE 4 — P/VP × P/L

-- Objetivo:
-- Calcular o indicador composto P/VP × P/L.

CREATE OR REPLACE TABLE workspace.gold.analise_acoes AS

SELECT

    -- SELECT:
    -- define as colunas que continuarão na tabela final.
    
    -- Mantemos todos os indicadores já calculados nos stages anteriores.

    codigo_acao,
    preco_atual,
    data_preco,

    pl,
    pvp,

    -- Multiplicação:
    
    -- Aqui estamos multiplicando P/VP pelo P/L.
    
    pvp * pl AS pvp_x_pl,

    valor_patrimonial,
    eps,
    margem_liquida,
    roe,
    ebitda,
    divida_total,
    caixa,
    divida_liquida,

    dividendos_medio_5_anos_reais,
    dy_medio_5_anos_pct,

    crescimento_lucro_cagr_5_anos_pct

FROM workspace.gold.analise_acoes;

In [0]:
%sql

-- VERIFICAÇÃO DO STAGE 4

-- Mostra os dois indicadores originais e o resultado da multiplicação para facilitar a conferência.

SELECT
    codigo_acao,
    pl,
    pvp,
    pvp_x_pl

FROM workspace.gold.analise_acoes

ORDER BY codigo_acao;

In [0]:
%sql

-- STAGE 5 — MARGEM LÍQUIDA E ROE

-- Esses dois indicadores já vieram da BRAPI e foram padronizados na camada Silver.

-- margem_liquida = profitMargins da BRAPI
-- roe            = returnOnEquity da BRAPI

-- Aqui apenas mostramos os valores atuais da Gold para conferência.

SELECT
    codigo_acao,
    margem_liquida,
    margem_liquida * 100 AS margem_liquida_pct,
    roe,
    roe * 100 AS roe_pct

FROM workspace.gold.analise_acoes

ORDER BY codigo_acao;

In [0]:
%sql

-- STAGE 6 — DÍVIDA LÍQUIDA / EBITDA


-- O objetivo é medir quantas vezes o EBITDA cobre a dívida líquida da empresa.

-- Fórmula:
-- Dívida Líquida / EBITDA = (Dívida Total - Caixa) / EBITDA

-- A dívida líquida já foi calculada no Stage 1.
-- O EBITDA também já veio pronto da BRAPI.

CREATE OR REPLACE TABLE workspace.gold.analise_acoes AS

SELECT
    *,
    
    CASE
        -- Se não houver EBITDA ou dívida líquida, não temos dados suficientes para calcular o indicador.
        WHEN ebitda IS NULL OR divida_liquida IS NULL THEN NULL
        
        -- Evita divisão por zero.
        WHEN ebitda = 0 THEN NULL
        
        ELSE divida_liquida / ebitda
    END AS divida_liquida_ebitda

FROM workspace.gold.analise_acoes;

In [0]:
%sql

-- VERIFICAÇÃO DO STAGE 6

SELECT
    codigo_acao,
    ebitda,
    divida_total,
    caixa,
    divida_liquida,
    divida_liquida_ebitda

FROM workspace.gold.analise_acoes

ORDER BY codigo_acao;

In [0]:
%sql
-- STAGE 7 — PREÇO TETO (BAZIN)

-- Fórmula definida no critério do projeto:
-- Preço Teto = Média dos dividendos dos 5 anos / Rendimento desejado

CREATE OR REPLACE TABLE workspace.gold.analise_acoes AS

SELECT
    *,
    
    CASE
        -- Sem média histórica de dividendos, não é possível calcular o preço teto.
        --O código aplica a fórmula caso o valor médio de dividendos não seja nulo.
        WHEN dividendos_medio_5_anos_reais IS NULL THEN NULL
        
        ELSE dividendos_medio_5_anos_reais / 0.06
    END AS preco_teto_bazin

FROM workspace.gold.analise_acoes;

In [0]:
%sql

-- VERIFICAÇÃO DO STAGE 7

SELECT
    codigo_acao,
    dividendos_medio_5_anos_reais,
    preco_teto_bazin,
    preco_atual

FROM workspace.gold.analise_acoes

ORDER BY codigo_acao;

In [0]:
%sql
-- STAGE 8 — VALOR JUSTO DE GRAHAM

-- Fórmula definida no critério do projeto:

-- Valor Justo = SQRT(22,5 × Lucro Por Ação × Valor Patrimonial por Ação)
-- SQRT() calcula a raiz quadrada no SQL.

-- Os valores de EPS (LPA) e valor patrimonial já vieram da BRAPI e estão armazenados na Gold.

CREATE OR REPLACE TABLE workspace.gold.analise_acoes AS

SELECT
    *,
    
    CASE
        -- Se faltar EPS ou valor patrimonial, não conseguimos calcular o valor justo.
        WHEN eps IS NULL OR valor_patrimonial IS NULL THEN NULL
        
        -- Evita calcular a raiz de um número negativo.
        WHEN 22.5 * eps * valor_patrimonial < 0 THEN NULL
        
        ELSE SQRT(
            22.5 * eps * valor_patrimonial
        )
    END AS valor_justo_graham

FROM workspace.gold.analise_acoes;

In [0]:
%sql

-- VERIFICAÇÃO DO STAGE 8

SELECT
    codigo_acao,
    preco_atual,
    eps,
    valor_patrimonial,
    valor_justo_graham

FROM workspace.gold.analise_acoes

ORDER BY codigo_acao;

In [0]:
%sql

-- STAGE 9 — TABELA FINAL DA ANÁLISE

-- A tabela Gold reúne os indicadores necessários para responder às perguntas de negócio do MVP.

-- Os indicadores abaixo já foram calculados nos stages anteriores. Aqui apenas organizamos a visão final.

CREATE OR REPLACE TABLE workspace.gold.analise_acoes AS

SELECT

    -- Identificação e preço
    codigo_acao,
    preco_atual,
    data_preco,

    -- Indicadores de valuation
    pl,
    pvp,
    pvp_x_pl,
    eps,
    valor_patrimonial,

    -- Rentabilidade
    margem_liquida,
    roe,

    -- Estrutura financeira
    ebitda,
    divida_total,
    caixa,
    divida_liquida,
    divida_liquida_ebitda,

    -- Proventos
    dividendos_medio_5_anos_reais,
    dy_medio_5_anos_pct,

    -- Crescimento
    crescimento_lucro_cagr_5_anos_pct,

    -- Métodos de preço
    preco_teto_bazin,
    valor_justo_graham

FROM workspace.gold.analise_acoes;

In [0]:
%sql

-- VERIFICAÇÃO DA TABELA GOLD FINAL

-- SELECT * traz todas as colunas da tabela.
-- ORDER BY organiza as ações pelo código.

SELECT *
FROM workspace.gold.analise_acoes
ORDER BY codigo_acao;

In [0]:
%sql

-- STAGE 10 — APLICAÇÃO DOS CRITÉRIOS


-- Criamos uma nova tabela para não modificar os indicadores originais da tabela analise_acoes.

-- A tabela analise_acoes contém os valores calculados.
-- A tabela criterios_acoes contém a interpretação desses valores segundo as regras definidas para o projeto.


CREATE OR REPLACE TABLE workspace.gold.criterios_acoes AS

SELECT

    -- Identificação
    codigo_acao,

    
    -- INDICADORES ORIGINAIS

    preco_atual,
    pl,
    pvp,
    pvp_x_pl,
    margem_liquida,
    roe,
    divida_liquida_ebitda,
    dy_medio_5_anos_pct,
    crescimento_lucro_cagr_5_anos_pct,
    preco_teto_bazin,
    valor_justo_graham,


    -- MARGEM DE SEGURANÇA DE GRAHAM

    -- 25% de margem de segurança significa utilizar apenas 75% do valor justo como preço máximo de compra.

    valor_justo_graham * 0.75 AS preco_max_graham_25,

 
    -- CRITÉRIO 1 — DIVIDEND YIELD

    CASE
        WHEN dy_medio_5_anos_pct > 6
        THEN 'ATENDE'
        WHEN dy_medio_5_anos_pct IS NULL
        THEN 'SEM DADO'
        ELSE 'NÃO ATENDE'
    END AS criterio_dy,


    -- CRITÉRIO 2 — CRESCIMENTO DO LUCRO

    CASE
        WHEN crescimento_lucro_cagr_5_anos_pct > 12
        THEN 'ATENDE'
        WHEN crescimento_lucro_cagr_5_anos_pct IS NULL
        THEN 'SEM DADO'
        ELSE 'NÃO ATENDE'
    END AS criterio_crescimento,

  
    -- CRITÉRIO 3 — P/L MENOR QUE O CRESCIMENTO

    CASE
        WHEN pl IS NULL
          OR crescimento_lucro_cagr_5_anos_pct IS NULL
        THEN 'SEM DADO'

        WHEN pl < crescimento_lucro_cagr_5_anos_pct
        THEN 'ATENDE'

        ELSE 'NÃO ATENDE'
    END AS criterio_pl_crescimento,


    -- CRITÉRIO 4 — P/VP × P/L

    CASE
        WHEN pvp_x_pl IS NULL
        THEN 'SEM DADO'

        WHEN pvp_x_pl <= 22.5
        THEN 'ATENDE'

        ELSE 'NÃO ATENDE'
    END AS criterio_pvp_x_pl,

   
    -- CRITÉRIO 5 — MARGEM LÍQUIDA

    CASE
        WHEN margem_liquida IS NULL
        THEN 'SEM DADO'

        WHEN margem_liquida >= 0.10
        THEN 'ATENDE'

        ELSE 'NÃO ATENDE'
    END AS criterio_margem,

    -- CRITÉRIO 6 — ROE

    CASE
        WHEN roe IS NULL
        THEN 'SEM DADO'

        WHEN roe >= 0.10
        THEN 'ATENDE'

        ELSE 'NÃO ATENDE'
    END AS criterio_roe,

    -- CRITÉRIO 7 — DÍVIDA LÍQUIDA / EBITDA

    CASE
        WHEN divida_liquida_ebitda IS NULL
        THEN 'SEM DADO'

        WHEN divida_liquida_ebitda < 5
        THEN 'ATENDE'

        ELSE 'NÃO ATENDE'
    END AS criterio_divida,

    -- CRITÉRIO 8 — PREÇO TETO BAZIN

    CASE
        WHEN preco_teto_bazin IS NULL
        THEN 'SEM DADO'

        WHEN preco_atual <= preco_teto_bazin
        THEN 'ATENDE'

        ELSE 'NÃO ATENDE'
    END AS criterio_bazin,

    -- CRITÉRIO 9 — VALOR JUSTO DE GRAHAM
    
    CASE
        WHEN valor_justo_graham IS NULL
        THEN 'SEM DADO'

        WHEN preco_atual <= valor_justo_graham * 0.75
        THEN 'ATENDE'

        ELSE 'NÃO ATENDE'
    END AS criterio_graham

FROM workspace.gold.analise_acoes;

In [0]:
%sql
-- VERIFICAÇÃO DOS CRITÉRIOS

SELECT *
FROM workspace.gold.criterios_acoes
ORDER BY codigo_acao;